In [0]:
from pyspark.ml.feature import VectorAssembler
from pyspark.sql.functions import when, col

# 1. Özellik tablosunu oku
features_path = "/Volumes/workspace/default/steam/steam_reviews_features"
df_features = spark.read.format("delta").load(features_path)

# 2. Hedef değişkeni oluştur (1.0 ve 0.0)
df_ml = df_features.withColumn("label", when(col("voted_up").cast("string").isin("1", "true"), 1.0).otherwise(0.0))

# 3. Vektörizasyon
feature_columns = ["review_length", "word_count", "is_voted", "has_positive_word", "has_negative_word"]
assembler = VectorAssembler(inputCols=feature_columns, outputCol="features")
final_data = assembler.transform(df_ml).select("features", "label")

# 4. Train/Test Split (%80 Eğitim, %20 Test)
train_data, test_data = final_data.randomSplit([0.8, 0.2], seed=42)

print(f"Dengeleme Öncesi Orijinal Eğitim Seti: {train_data.count()} satır")

# ==========================================
# 5. SINIF DENGESİZLİĞİNİ ÇÖZME (UNDERSAMPLING)
# ==========================================
print("\nEğitim verisi dengeleniyor... Lütfen bekleyin.")

# Sınıfları ayır
negatif_df = train_data.filter(col("label") == 0.0)
olumlu_df = train_data.filter(col("label") == 1.0)

negatif_sayisi = negatif_df.count()
olumlu_sayisi = olumlu_df.count()

# Olumlu yorum sayısını, negatif yorum sayısı kadar rastgele küçültüyoruz
oran = negatif_sayisi / olumlu_sayisi
olumlu_alt_kume = olumlu_df.sample(withReplacement=False, fraction=oran, seed=42)

# Eşitlenmiş verileri birleştir
balanced_train_data = negatif_df.unionByName(olumlu_alt_kume)

print(f"YENİ Dengeli Eğitim Seti (Balanced Train): {balanced_train_data.count()} satır (%50 Olumlu - %50 Olumsuz)")
print(f"Test Seti (Gerçek Hayat Simülasyonu): {test_data.count()} satır")

Dengeleme Öncesi Orijinal Eğitim Seti: 5133697 satır

Eğitim verisi dengeleniyor... Lütfen bekleyin.
YENİ Dengeli Eğitim Seti (Balanced Train): 1850067 satır (%50 Olumlu - %50 Olumsuz)
Test Seti (Gerçek Hayat Simülasyonu): 1283409 satır


In [0]:
import mlflow
import mlflow.spark
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier, DecisionTreeClassifier, GBTClassifier, LinearSVC
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator

# 1. Güçlendirilmiş Parametrelerle 5 Modeli Tanımlayalım
modeller = [
    ("Logistic Regression", LogisticRegression(featuresCol="features", labelCol="label", maxIter=20)),
    ("Random Forest", RandomForestClassifier(featuresCol="features", labelCol="label", numTrees=50, maxDepth=7, seed=42)),
    ("Decision Tree", DecisionTreeClassifier(featuresCol="features", labelCol="label", maxDepth=7, seed=42)),
    ("Gradient Boosted Trees", GBTClassifier(featuresCol="features", labelCol="label", maxIter=20, maxDepth=7, seed=42)),
    ("Linear SVC", LinearSVC(featuresCol="features", labelCol="label", maxIter=20))
]

print("Dengeli veri seti ile 5 model sırasıyla eğitiliyor...\n")

# Değerlendiricileri (Evaluators) hazırlayalım
multi_evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction")
binary_evaluator = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC")

# 2. Döngü ile Eğitim ve MLflow Loglaması
for model_adi, model_algoritmasi in modeller:
    with mlflow.start_run(run_name=model_adi.replace(" ", "_")):
        print(f">>> {model_adi} model eğitimi başladı...")
        
        # Modeli DENGELİ VERİ (balanced_train_data) ile eğit!
        model = model_algoritmasi.fit(balanced_train_data)
        
        # Tahmini Orijinal Test Verisi ile yap (Gerçek hayat)
        predictions = model.transform(test_data)
        
        # Metrikleri Hesapla
        accuracy = multi_evaluator.evaluate(predictions, {multi_evaluator.metricName: "accuracy"})
        f1_score = multi_evaluator.evaluate(predictions, {multi_evaluator.metricName: "f1"})
        auc_roc = binary_evaluator.evaluate(predictions)
        
        print(f"    Sonuçlar -> Accuracy: {accuracy:.4f} | F1-Score: {f1_score:.4f} | AUC-ROC: {auc_roc:.4f}\n")
        
        # MLflow'a Logla
        mlflow.log_param("model_type", model_adi)
        mlflow.log_param("training_type", "Undersampled Balanced Data")
        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("f1_score", f1_score)
        mlflow.log_metric("auc_roc", auc_roc)
        
        # Modeli Kaydet
        mlflow.spark.log_model(model, f"{model_adi.replace(' ', '_').lower()}_model", dfs_tmpdir="/Volumes/workspace/default/steam/tmp")

print("MÜKEMMEL! Tüm modeller dengeli veri setiyle eğitildi ve MLflow'a başarıyla kaydedildi.")

Dengeli veri seti ile 5 model sırasıyla eğitiliyor...

>>> Logistic Regression model eğitimi başladı...
    Sonuçlar -> Accuracy: 0.6946 | F1-Score: 0.7232 | AUC-ROC: 0.7023



2026/05/12 22:03:14 WARNING mlflow.utils.requirements_utils: Found pyspark version (4.1.0+databricks.connect.18.0.5) contains a local version label (+databricks.connect.18.0.5). MLflow logged a pip requirement for this package as 'pyspark==4.1.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/05/12 22:03:18 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /local_disk0/user_tmp_data/spark-e91ebf1d-3b9e-43bf-89f4-ef/tmpj8uir4jt/model, flavor: spark). Fall back to return ['pyspark==4.1.0']. Set logging level to DEBUG to see the full traceback. 
2026/05/12 22:03:18 INFO mlflow.models.model: Model logged without a signature. Signatures are required for Databricks UC model registry as they validate model inputs and denote the expected schema of model outputs. Please set `input_example` parameter when l

>>> Random Forest model eğitimi başladı...
    Sonuçlar -> Accuracy: 0.6046 | F1-Score: 0.6501 | AUC-ROC: 0.7074



2026/05/12 22:04:37 WARNING mlflow.utils.requirements_utils: Found pyspark version (4.1.0+databricks.connect.18.0.5) contains a local version label (+databricks.connect.18.0.5). MLflow logged a pip requirement for this package as 'pyspark==4.1.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/05/12 22:04:39 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /local_disk0/user_tmp_data/spark-e91ebf1d-3b9e-43bf-89f4-ef/tmpy23ett0c/model, flavor: spark). Fall back to return ['pyspark==4.1.0']. Set logging level to DEBUG to see the full traceback. 
2026/05/12 22:04:39 INFO mlflow.models.model: Model logged without a signature. Signatures are required for Databricks UC model registry as they validate model inputs and denote the expected schema of model outputs. Please set `input_example` parameter when l

>>> Decision Tree model eğitimi başladı...
    Sonuçlar -> Accuracy: 0.5855 | F1-Score: 0.6327 | AUC-ROC: 0.6823



...

>>> Gradient Boosted Trees model eğitimi başladı...
    Sonuçlar -> Accuracy: 0.6003 | F1-Score: 0.6463 | AUC-ROC: 0.7094



2026/05/12 22:07:27 WARNING mlflow.utils.requirements_utils: Found pyspark version (4.1.0+databricks.connect.18.0.5) contains a local version label (+databricks.connect.18.0.5). MLflow logged a pip requirement for this package as 'pyspark==4.1.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/05/12 22:07:29 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /local_disk0/user_tmp_data/spark-e91ebf1d-3b9e-43bf-89f4-ef/tmpt2nfsza2/model, flavor: spark). Fall back to return ['pyspark==4.1.0']. Set logging level to DEBUG to see the full traceback. 
2026/05/12 22:07:29 INFO mlflow.models.model: Model logged without a signature. Signatures are required for Databricks UC model registry as they validate model inputs and denote the expected schema of model outputs. Please set `input_example` parameter when l

>>> Linear SVC model eğitimi başladı...
    Sonuçlar -> Accuracy: 0.7409 | F1-Score: 0.7531 | AUC-ROC: 0.6995



2026/05/12 22:08:18 WARNING mlflow.utils.requirements_utils: Found pyspark version (4.1.0+databricks.connect.18.0.5) contains a local version label (+databricks.connect.18.0.5). MLflow logged a pip requirement for this package as 'pyspark==4.1.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/05/12 22:08:20 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /local_disk0/user_tmp_data/spark-e91ebf1d-3b9e-43bf-89f4-ef/tmp_6snn8s7/model, flavor: spark). Fall back to return ['pyspark==4.1.0']. Set logging level to DEBUG to see the full traceback. 
2026/05/12 22:08:20 INFO mlflow.models.model: Model logged without a signature. Signatures are required for Databricks UC model registry as they validate model inputs and denote the expected schema of model outputs. Please set `input_example` parameter when l

MÜKEMMEL! Tüm modeller dengeli veri setiyle eğitildi ve MLflow'a başarıyla kaydedildi.
